# Brain Section Segmentation with a Small CNN

This notebook trains a small **Convolutional Neural Network (CNN)** to segment
grey matter from brain sections, and shows you what the network is doing along
the way.

It has two parts:

- **Part 1 — Demo data.** Runs immediately on a small synthetic dataset so you
  can see the whole pipeline work end to end and understand each piece.
- **Part 2 — Your real data.** After Part 1, you'll upload real brain images and
  the segmentations you drew by hand, and run the same network on them.

**How to use a notebook:** click a cell and press **Shift+Enter** to run it. Run
the cells **in order** from top to bottom. Text cells (like this one) explain
what's happening; code cells do the work.

> **Tip:** In Colab, go to **Runtime → Change runtime type** and pick **GPU** if
> it's available — training is faster on a GPU. It will still work on CPU.

## Setup: install and import the tools

This installs the libraries we need (`nibabel` for reading brain images) and
imports everything. Run it first; it takes a few seconds.

In [ ]:
# Colab already has torch, numpy, matplotlib. We just need nibabel.
!pip install nibabel --quiet

import os, glob
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# Use a GPU if the runtime has one, otherwise the CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", device)
torch.manual_seed(0)

---
# Part 1 — Demo data

## What are we trying to do?

**Segmentation** means labelling every pixel of an image. Here the question for
each pixel is: *"is this grey matter, yes or no?"* So the answer to a whole
image is another image of the same size, full of 0s and 1s — a **mask**.

A **neural network** is a function with many adjustable numbers inside it (its
**weights**). We show it an image, it predicts a mask, we measure how wrong the
mask is, and we nudge the weights to be a little less wrong. Repeat thousands of
times and the network *learns* to segment. That loop — **predict → measure error
→ nudge** — is the whole idea of **training**.

## Make some demo data

Real brain images come later. For now we generate a small synthetic dataset with
the **same folder layout as the real data**:

- `demo_data/images/<name>_0000.nii.gz` — the raw section
- `demo_data/label/<name>.nii.gz` — the hand-drawn grey-matter label

Each synthetic "section" is a bright cortical ribbon (the grey matter) around a
white-matter core, on a dark background. Simple, but enough to learn on.

In [ ]:
os.makedirs("demo_data/images", exist_ok=True)
os.makedirs("demo_data/label", exist_ok=True)

def make_section(seed):
    r = np.random.default_rng(seed)
    H = W = 96
    img = np.full((H, W), 0.15)                    # dark background
    yy, xx = np.ogrid[:H, :W]
    cx, cy = W//2 + r.integers(-6, 6), H//2 + r.integers(-6, 6)
    rad = r.integers(30, 38)
    dist = np.sqrt((xx - cx)**2 + (yy - cy)**2)
    img[dist < rad] = 0.5                           # white matter
    ribbon = (dist < rad) & (dist > rad - 7)
    img[ribbon] = 0.8                               # cortex = grey matter
    img = np.clip(img + r.normal(0, 0.05, (H, W)), 0, 1).astype(np.float32)
    return img, ribbon.astype(np.uint8)

for i in range(12):
    img, lab = make_section(i)
    nib.save(nib.Nifti1Image(img, np.eye(4)), f"demo_data/images/sec{i:02d}_0000.nii.gz")
    nib.save(nib.Nifti1Image(lab, np.eye(4)), f"demo_data/label/sec{i:02d}.nii.gz")

print("Wrote 12 image/label pairs into demo_data/")

## Load the data

Each image file ends in `_0000.nii.gz`; its matching label has the same name
without that suffix. We load every pair and stack them into **tensors** (the
array type PyTorch uses).

The shape `(N, 1, H, W)` means: N images, 1 channel each (greyscale), height H,
width W. That extra "1" is the **channel** dimension — PyTorch convolutions
always expect one, even for greyscale.

In [ ]:
IMAGES_DIR = "demo_data/images"
LABELS_DIR = "demo_data/label"

def load_pair(image_path, labels_dir):
    name = os.path.basename(image_path).replace("_0000.nii.gz", "")
    lab_path = os.path.join(labels_dir, name + ".nii.gz")
    img = np.squeeze(nib.load(image_path).get_fdata()).astype(np.float32)
    lab = np.squeeze(nib.load(lab_path).get_fdata()).astype(np.float32)
    return img, lab

def load_dataset(images_dir, labels_dir):
    paths = sorted(glob.glob(os.path.join(images_dir, "*_0000.nii.gz")))
    imgs, labs = [], []
    for p in paths:
        img, lab = load_pair(p, labels_dir)
        imgs.append(img); labs.append(lab)
    X = torch.tensor(np.stack(imgs))[:, None, :, :]   # (N, 1, H, W)
    Y = torch.tensor(np.stack(labs))[:, None, :, :]
    return X, Y

X, Y = load_dataset(IMAGES_DIR, LABELS_DIR)
print("Loaded", X.shape[0], "sections of size", tuple(X.shape[2:]))

### Always look at your data first

Before training anything, plot a few images with their labels. If these look
wrong, nothing downstream will work.

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(11, 5.5))
for i in range(4):
    ax[0, i].imshow(X[i, 0], cmap="gray"); ax[0, i].set_title(f"image {i}"); ax[0, i].axis("off")
    ax[1, i].imshow(Y[i, 0], cmap="gray"); ax[1, i].set_title(f"label {i} (GM)"); ax[1, i].axis("off")
plt.tight_layout(); plt.show()

## Why a *convolutional* network?

A plain ("dense") network gives every pixel its own separate weight and treats
pixels as unrelated. Two problems:

1. **No sense of neighbourhood.** Whether a pixel is grey matter depends on the
   pixels *around* it. A dense network can't see that.
2. **Too many weights.** A 96×96 image has ~9000 pixels; a dense layer would
   need thousands of weights for every output pixel.

A **convolution** fixes both. It learns a small **filter** (say 3×3) and slides
that *same* filter across the whole image, always looking at each pixel together
with its neighbours. So it naturally uses local context, and reuses the same few
weights everywhere. That's why CNNs are the standard tool for images.

Our network stacks **three convolutional layers**:
- layer 1: turns the 1-channel image into 8 **feature maps** (each = one learned
  filter's response),
- layer 2: mixes those 8 into 8 richer ones,
- layer 3: collapses to a single map — the score for "how grey-matter-like is
  this pixel?"

Between layers we apply a **ReLU** (keep positives, zero out negatives). Without
a non-linearity like ReLU, stacking layers would be pointless — three linear
layers in a row equal just one. The ReLU is what lets depth add power.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(8, 8, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(8, 1, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.conv3(x)          # one score ("logit") per pixel
        return x

model = SmallCNN().to(device)
n_params = sum(p.numel() for p in model.parameters())
print("The network has", n_params, "learnable weights")

## Split into training and test sets

We must judge the network on data it did **not** learn from — otherwise it could
just memorise the answers and look better than it really is. So we hold a few
sections back as a **test set** and train only on the rest.

In [ ]:
n_test = max(1, X.shape[0] // 4)
X_train, Y_train = X[:-n_test].to(device), Y[:-n_test].to(device)
X_test,  Y_test  = X[-n_test:].to(device), Y[-n_test:].to(device)
print("Training on", X_train.shape[0], "sections, testing on", X_test.shape[0])

## Training: loss, gradient descent, and class imbalance

The network's raw output is a grid of numbers called **logits** (not yet 0/1).
A **loss function** measures how wrong they are — one number, big when wrong,
small when right. We use **binary cross-entropy** (`BCEWithLogitsLoss`).

**Class imbalance — an important gotcha.** Grey matter is only a small fraction
of each image; most pixels are background or white matter. A lazy network could
get a *low* loss just by predicting "not grey matter" everywhere — right most of
the time, barely punished for missing the rare GM pixels. The fix is
**`pos_weight`**: it tells the loss to care *more* about the rare grey-matter
pixels, in proportion to how rare they are. This one change is often the
difference between a network that cheats and one that actually finds the cortex.

**Gradient descent** is the learning loop itself:
1. run images through the network (**forward pass**),
2. compute the loss,
3. compute the **gradient** — which way to nudge each weight to lower the loss
   (`loss.backward()`),
4. take a small step that way (`optimizer.step()`).

Repeat for many **epochs** (full passes over the data) and the loss shrinks.

In [ ]:
def train(model, X, Y, epochs=150, lr=0.01):
    gm_fraction = Y.mean()
    pos_weight = (1 - gm_fraction) / gm_fraction     # weight the rare GM class up
    print(f"Grey matter is {gm_fraction.item():.1%} of pixels "
          f"-> weighting it x{pos_weight.item():.1f}")
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(epochs):
        optimizer.zero_grad()
        logits = model(X)
        loss = loss_fn(logits, Y)
        loss.backward()
        optimizer.step()
        history.append(loss.item())
        if epoch % 20 == 0 or epoch == epochs - 1:
            print(f"  epoch {epoch:3d}   loss {loss.item():.4f}")
    return history

history = train(model, X_train, Y_train)

### The loss curve

Watching the loss go **down** over epochs is how you see learning happen. A flat
curve means the network isn't learning; a wildly jumping curve means the steps
are too big (lower the learning rate).

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history)
plt.xlabel("epoch"); plt.ylabel("loss (lower = better)")
plt.title("Training loss over time"); plt.show()

## How good is it? Predictions and the Dice score

To turn the network's logits into a mask, we apply a **sigmoid** (squashes each
score to a probability between 0 and 1) and then threshold at 0.5.

We score the result with the **Dice score**: `2 * overlap / (predicted + true)`.
It runs from 0 (no overlap) to 1 (perfect). We compute it on the **test** set —
the sections the network never trained on.

In [ ]:
def predict_mask(model, x):
    model.eval()
    with torch.no_grad():
        prob = torch.sigmoid(model(x))
    return (prob > 0.5).float()

def dice_score(pred, truth):
    pred = pred.bool(); truth = truth.bool()
    overlap = (pred & truth).sum().item()
    denom = pred.sum().item() + truth.sum().item()
    return 2.0 * overlap / denom if denom > 0 else float("nan")

dices = []
for i in range(X_test.shape[0]):
    pred = predict_mask(model, X_test[i:i+1])
    d = dice_score(pred, Y_test[i:i+1])
    dices.append(d)
    print(f"test section {i}: Dice = {d:.3f}")
print(f"\nMean test Dice = {np.nanmean(dices):.3f}")

### See the predictions

Input, the true hand label, and the network's prediction, side by side. On this
easy demo data the network should score very high (~0.99). Real data will be
messier and score lower — that's expected, not failure.

In [ ]:
n = X_test.shape[0]
plt.figure(figsize=(9, 3*n))
for i in range(n):
    pred = predict_mask(model, X_test[i:i+1])
    d = dice_score(pred, Y_test[i:i+1])
    plt.subplot(n, 3, i*3+1); plt.imshow(X_test[i,0].cpu(), cmap="gray")
    plt.title("input"); plt.axis("off")
    plt.subplot(n, 3, i*3+2); plt.imshow(Y_test[i,0].cpu(), cmap="gray")
    plt.title("true label"); plt.axis("off")
    plt.subplot(n, 3, i*3+3); plt.imshow(pred[0,0].cpu(), cmap="gray")
    plt.title(f"prediction (Dice={d:.3f})"); plt.axis("off")
plt.tight_layout(); plt.show()

## Peek *inside* the network: feature maps

This shows the 8 feature maps the **first** convolutional layer produces for one
image — each is what one learned 3×3 filter "lit up" on. Some respond to edges,
some to the bright ribbon, some to the interior. This is the clearest picture of
what "learning filters" actually means: the network built these detectors by
itself, just from being trained to segment.

In [ ]:
def show_feature_maps(model, x):
    model.eval()
    with torch.no_grad():
        first = model.relu(model.conv1(x))     # (1, 8, H, W)
    maps = first[0].cpu().numpy()
    plt.figure(figsize=(13, 3))
    plt.subplot(1, 9, 1); plt.imshow(x[0,0].cpu(), cmap="gray")
    plt.title("input"); plt.axis("off")
    for i in range(8):
        plt.subplot(1, 9, i+2); plt.imshow(maps[i], cmap="viridis")
        plt.title(f"filter {i+1}", fontsize=8); plt.axis("off")
    plt.tight_layout(); plt.show()

show_feature_maps(model, X_test[0:1])

### What you just saw (Part 1 recap)

- A CNN learns **filters** that detect local patterns, and stacks them to build
  up to a per-pixel decision.
- **Training** = predict → measure loss → nudge weights, repeated over epochs;
  the loss curve shows it working.
- **Class imbalance** can make a network cheat; `pos_weight` fixes it.
- We judge quality on a **held-out test set** with the **Dice score**.

Now let's run the same pipeline on **real** brain sections.

---
# Part 2 — Your real data

Now you'll swap in real brain images and the segmentations drawn by hand. The
code is the same as Part 1 — only the data folders change.

### Step 1: upload your data

Get your files into Colab in the **same layout** as the demo:

```
real_data/images/<name>_0000.nii.gz    (raw sections)
real_data/label/<name>.nii.gz          (your manual GM labels)
```

Two easy ways to do this:

**Option A — Google Drive (recommended for many files).** Mount your Drive and
point the folders at wherever your data lives. Run the cell below and follow the
prompt to connect your Drive.

**Option B — direct upload (fine for a few files).** Use the Files panel on the
left (the folder icon), create `real_data/images` and `real_data/label`, and
drag your files in.

In [ ]:
# OPTION A: mount Google Drive (skip if you're using direct upload)
from google.colab import drive
drive.mount('/content/drive')

# Then set these to point at your data inside Drive, e.g.:
# REAL_IMAGES_DIR = "/content/drive/MyDrive/brain_data/images"
# REAL_LABELS_DIR = "/content/drive/MyDrive/brain_data/label"
REAL_IMAGES_DIR = "real_data/images"   # <-- change to your path
REAL_LABELS_DIR = "real_data/label"    # <-- change to your path
print("images dir:", REAL_IMAGES_DIR)
print("labels dir:", REAL_LABELS_DIR)

### Step 2: load and look at the real data

Same loader as before. **Always plot a few first** to confirm the images and
labels line up. If the real sections are different sizes from each other, they'll
need resizing to a common size before training — ask if you hit that.

In [ ]:
X_real, Y_real = load_dataset(REAL_IMAGES_DIR, REAL_LABELS_DIR)
print("Loaded", X_real.shape[0], "real sections of size", tuple(X_real.shape[2:]))

k = min(4, X_real.shape[0])
fig, ax = plt.subplots(2, k, figsize=(3*k, 5.5))
for i in range(k):
    ax[0, i].imshow(X_real[i, 0], cmap="gray"); ax[0, i].set_title(f"image {i}"); ax[0, i].axis("off")
    ax[1, i].imshow(Y_real[i, 0], cmap="gray"); ax[1, i].set_title(f"label {i}"); ax[1, i].axis("off")
plt.tight_layout(); plt.show()

### Step 3: train on the real data

We make a fresh network and train it the same way. With real data you may need
**more epochs**, and results will be lower than the demo — real cortex is harder
than a clean synthetic ring.

In [ ]:
n_test = max(1, X_real.shape[0] // 4)
Xtr, Ytr = X_real[:-n_test].to(device), Y_real[:-n_test].to(device)
Xte, Yte = X_real[-n_test:].to(device), Y_real[-n_test:].to(device)
print("Training on", Xtr.shape[0], "real sections, testing on", Xte.shape[0])

real_model = SmallCNN().to(device)
real_history = train(real_model, Xtr, Ytr, epochs=200)

plt.figure(figsize=(6, 4)); plt.plot(real_history)
plt.xlabel("epoch"); plt.ylabel("loss"); plt.title("Training loss (real data)"); plt.show()

### Step 4: evaluate and view predictions on real sections

In [ ]:
dices = []
for i in range(Xte.shape[0]):
    pred = predict_mask(real_model, Xte[i:i+1])
    dices.append(dice_score(pred, Yte[i:i+1]))
print(f"Mean test Dice on real data = {np.nanmean(dices):.3f}")

n = min(3, Xte.shape[0])
plt.figure(figsize=(9, 3*n))
for i in range(n):
    pred = predict_mask(real_model, Xte[i:i+1])
    d = dice_score(pred, Yte[i:i+1])
    plt.subplot(n, 3, i*3+1); plt.imshow(Xte[i,0].cpu(), cmap="gray"); plt.title("input"); plt.axis("off")
    plt.subplot(n, 3, i*3+2); plt.imshow(Yte[i,0].cpu(), cmap="gray"); plt.title("your manual label"); plt.axis("off")
    plt.subplot(n, 3, i*3+3); plt.imshow(pred[0,0].cpu(), cmap="gray"); plt.title(f"CNN (Dice={d:.3f})"); plt.axis("off")
plt.tight_layout(); plt.show()

### Step 5: see what the real-data network learned

In [ ]:
show_feature_maps(real_model, Xte[0:1])

### Discuss

- How does the CNN's Dice on real data compare to the **Otsu** results from the
  earlier exercises? Better, worse, similar?
- Where does the network still get it wrong? Look at the prediction vs your
  manual label — are the mistakes at the cortex boundary, or somewhere specific?
- What might help: more training sections, more epochs, or a bigger network?
  (The next stage of the project — a full U-Net like nnU-Net — is a much bigger
  version of exactly what you built here.)